# 02 - CICIDS2017: Multi-Class Benchmark (ML vs. Deep Learning)

## Objective
Conduct a comparative evaluation on the clean, balanced CICIDS2017 subset (`CICIDS_clean_subset.parquet`):
- **169,686 rows**, 44 clean features, 7 classes.
- Realistic ~59% Benign : 41% Attacks ratio.

### Models Evaluated (Phase 2 Comparative Study):
1. **Random Forest (Balanced)** — Classic Bagging Tree Ensemble
2. **LightGBM (Balanced)** — Gradient Boosted Decision Trees
3. **Deep Residual MLP (PyTorch)** — Multi-Layer Perceptron with Residual Skip Connections, BatchNorm & Dropout
4. **1D-CNN (PyTorch)** — Fast 1D Convolutional Neural Network for local tabular flow feature interactions

### Metrics Tracked:
- **Train vs. Test Accuracy** (Diagnoses Overfitting vs. Generalization gap)
- **Weighted F1 & Macro F1** (Fair evaluation across majority and rare classes)
- **Training Time (s) & Inference Latency (ms/sample)** (Operational efficiency)

In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

Using compute device: cpu


## 1. Load Clean Subset & Preprocess

In [2]:
t0 = time.time()
df = pd.read_parquet('../Datasets/CICIDS_clean_subset.parquet')
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns in {time.time()-t0:.2f} seconds!")

X = df.drop(columns=['Label'])
y = df['Label']

# Target Encoding
label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)
classes = list(label_encoder.classes_)
num_classes = len(classes)
print(f"Target classes ({num_classes}): {classes}")

# Stratified Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=SEED
)

# StandardScaler with clipping to [-10, 10] (crucial for neural network stability)
scaler = StandardScaler()
X_train_scaled = np.clip(scaler.fit_transform(X_train), -10, 10)
X_test_scaled = np.clip(scaler.transform(X_test), -10, 10)

print(f"Train Shape: {X_train_scaled.shape} | Test Shape: {X_test_scaled.shape}")

Loaded 169,686 rows and 45 columns in 0.10 seconds!
Target classes (7): ['BENIGN', 'Bot', 'Brute Force', 'DDoS', 'DoS', 'PortScan', 'Web Attack']
Train Shape: (135748, 44) | Test Shape: (33938, 44)


## 2. Machine Learning Baselines (Random Forest & LightGBM)

In [3]:
ml_models = {
    "Random Forest (Balanced)": RandomForestClassifier(
        n_estimators=100,
        max_depth=16,
        min_samples_leaf=10,
        class_weight='balanced_subsample',
        random_state=SEED,
        n_jobs=-1
    ),
    "LightGBM (Balanced)": LGBMClassifier(
        n_estimators=150,
        max_depth=8,
        learning_rate=0.05,
        is_unbalance=True,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )
}

benchmark_results = []

for name, model in ml_models.items():
    print(f"--> Training {name}...")
    start_train = time.time()
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - start_train
    
    # Inference & Latency measurement
    start_inf = time.time()
    test_preds = model.predict(X_test_scaled)
    inf_time = (time.time() - start_inf) / len(X_test_scaled) * 1000  # ms per sample
    
    train_preds = model.predict(X_train_scaled)
    
    train_acc = accuracy_score(y_train, train_preds)
    test_acc = accuracy_score(y_test, test_preds)
    gap = train_acc - test_acc
    
    prec = precision_score(y_test, test_preds, average='weighted', zero_division=0)
    rec = recall_score(y_test, test_preds, average='weighted', zero_division=0)
    w_f1 = f1_score(y_test, test_preds, average='weighted', zero_division=0)
    m_f1 = f1_score(y_test, test_preds, average='macro', zero_division=0)
    
    benchmark_results.append({
        "Model": name,
        "Type": "Machine Learning",
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Overfit Gap": gap,
        "Weighted F1": w_f1,
        "Macro F1": m_f1,
        "Train Time (s)": train_time,
        "Inference (ms/sample)": inf_time
    })
    print(f"    Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | Overfit Gap: {gap:.4f}")
    print(f"    Weighted F1: {w_f1:.4f} | Macro F1: {m_f1:.4f} | Time: {train_time:.1f}s")

print("ML Baselines complete!")

--> Training Random Forest (Balanced)...
    Train Acc: 0.9928 | Test Acc: 0.9922 | Overfit Gap: 0.0007
    Weighted F1: 0.9926 | Macro F1: 0.9579 | Time: 4.6s
--> Training LightGBM (Balanced)...
    Train Acc: 0.9996 | Test Acc: 0.9987 | Overfit Gap: 0.0009
    Weighted F1: 0.9987 | Macro F1: 0.9931 | Time: 5.3s
ML Baselines complete!


## 3. Deep Learning Architectures in PyTorch

In [4]:
# Prepare PyTorch Tensors and DataLoaders
BATCH_SIZE = 512

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
eval_train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

criterion = nn.CrossEntropyLoss()

def predict_dl(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for bx, _ in loader:
            bx = bx.to(device)
            preds = torch.argmax(model(bx), dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

### Model 1: Deep Residual MLP Architecture

In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout_rate=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim)
        )
        self.relu = nn.ReLU()
        
    def forward(self, x):
        return self.relu(x + self.block(x))

class DeepResidualMLP(nn.Module):
    def __init__(self, input_dim=44, num_classes=7):
        super().__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.res1 = ResidualBlock(256, dropout_rate=0.2)
        self.down = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.out_proj = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = self.in_proj(x)
        x = self.res1(x)
        x = self.down(x)
        return self.out_proj(x)

mlp_model = DeepResidualMLP(input_dim=44, num_classes=num_classes).to(device)
print(mlp_model)

DeepResidualMLP(
  (in_proj): Sequential(
    (0): Linear(in_features=44, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
  )
  (res1): ResidualBlock(
    (block): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (relu): ReLU()
  )
  (down): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
  )
  (out_proj): Linear(

### Train Deep Residual MLP

In [6]:
def train_dl_model(model, train_loader, test_loader, criterion, epochs=8, lr=0.003):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
    
    best_macro_f1 = 0.0
    best_weights = None
    start_time = time.time()
    
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(by)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Validation
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for bx, by in test_loader:
                bx = bx.to(device)
                preds = torch.argmax(model(bx), dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(by.numpy())
                
        val_acc = accuracy_score(all_targets, all_preds)
        val_macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
        scheduler.step(val_macro_f1)
        
        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_weights = model.state_dict().copy()
            
        print(f"Epoch {epoch:02d}/{epochs:02d} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_macro_f1:.4f}")
        
    total_train_time = time.time() - start_time
    model.load_state_dict(best_weights)
    return model, total_train_time

print("--> Training Deep Residual MLP...")
mlp_model, mlp_train_time = train_dl_model(mlp_model, train_loader, test_loader, criterion, epochs=8, lr=0.003)

# Evaluate MLP
start_inf = time.time()
mlp_test_preds = predict_dl(mlp_model, test_loader)
mlp_inf_time = (time.time() - start_inf) / len(X_test_scaled) * 1000

mlp_train_preds = predict_dl(mlp_model, eval_train_loader)

mlp_train_acc = accuracy_score(y_train, mlp_train_preds)
mlp_test_acc = accuracy_score(y_test, mlp_test_preds)
mlp_gap = mlp_train_acc - mlp_test_acc
mlp_wf1 = f1_score(y_test, mlp_test_preds, average="weighted", zero_division=0)
mlp_mf1 = f1_score(y_test, mlp_test_preds, average="macro", zero_division=0)

benchmark_results.append({
    "Model": "Deep Residual MLP",
    "Type": "Deep Learning",
    "Train Accuracy": mlp_train_acc,
    "Test Accuracy": mlp_test_acc,
    "Overfit Gap": mlp_gap,
    "Weighted F1": mlp_wf1,
    "Macro F1": mlp_mf1,
    "Train Time (s)": mlp_train_time,
    "Inference (ms/sample)": mlp_inf_time
})
print(f"Deep Residual MLP - Acc: {mlp_test_acc:.4f} | Macro F1: {mlp_mf1:.4f} | Gap: {mlp_gap:.4f}")

--> Training Deep Residual MLP...
Epoch 01/08 | Loss: 0.1820 | Val Acc: 0.9711 | Val Macro F1: 0.9050
Epoch 02/08 | Loss: 0.0931 | Val Acc: 0.9737 | Val Macro F1: 0.9268
Epoch 03/08 | Loss: 0.0796 | Val Acc: 0.9732 | Val Macro F1: 0.9264
Epoch 04/08 | Loss: 0.0737 | Val Acc: 0.9746 | Val Macro F1: 0.9323
Epoch 05/08 | Loss: 0.0705 | Val Acc: 0.9758 | Val Macro F1: 0.9298
Epoch 06/08 | Loss: 0.0659 | Val Acc: 0.9751 | Val Macro F1: 0.9371
Epoch 07/08 | Loss: 0.0646 | Val Acc: 0.9786 | Val Macro F1: 0.9410
Epoch 08/08 | Loss: 0.0638 | Val Acc: 0.9717 | Val Macro F1: 0.9311
Deep Residual MLP - Acc: 0.9717 | Macro F1: 0.9311 | Gap: -0.0002


### Model 2: Fast 1D-CNN Architecture (Flow Convolution)

In [7]:
class Fast1DCNN(nn.Module):
    def __init__(self, in_features=44, num_classes=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        return self.net(x.unsqueeze(1))

cnn_model = Fast1DCNN(in_features=44, num_classes=num_classes).to(device)
print(cnn_model)

Fast1DCNN(
  (net): Sequential(
    (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): AdaptiveAvgPool1d(output_size=1)
    (8): Flatten(start_dim=1, end_dim=-1)
    (9): Linear(in_features=64, out_features=32, bias=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=32, out_features=7, bias=True)
  )
)


### Train 1D-CNN

In [ ]:
print("--> Training Fast 1D-CNN...")
cnn_model, cnn_train_time = train_dl_model(cnn_model, train_loader, test_loader, criterion, epochs=8, lr=0.003)

# Evaluate 1D-CNN
start_inf = time.time()
cnn_test_preds = predict_dl(cnn_model, test_loader)
cnn_inf_time = (time.time() - start_inf) / len(X_test_scaled) * 1000

cnn_train_preds = predict_dl(cnn_model, eval_train_loader)

cnn_train_acc = accuracy_score(y_train, cnn_train_preds)
cnn_test_acc = accuracy_score(y_test, cnn_test_preds)
cnn_gap = cnn_train_acc - cnn_test_acc
cnn_wf1 = f1_score(y_test, cnn_test_preds, average="weighted", zero_division=0)
cnn_mf1 = f1_score(y_test, cnn_test_preds, average="macro", zero_division=0)

benchmark_results.append({
    "Model": "1D-CNN",
    "Type": "Deep Learning",
    "Train Accuracy": cnn_train_acc,
    "Test Accuracy": cnn_test_acc,
    "Overfit Gap": cnn_gap,
    "Weighted F1": cnn_wf1,
    "Macro F1": cnn_mf1,
    "Train Time (s)": cnn_train_time,
    "Inference (ms/sample)": cnn_inf_time
})
print(f"1D-CNN - Acc: {cnn_test_acc:.4f} | Macro F1: {cnn_mf1:.4f} | Gap: {cnn_gap:.4f}")

--> Training Fast 1D-CNN...
Epoch 01/08 | Loss: 0.5889 | Val Acc: 0.9154 | Val Macro F1: 0.5231
Epoch 02/08 | Loss: 0.2331 | Val Acc: 0.9259 | Val Macro F1: 0.5956
Epoch 03/08 | Loss: 0.1808 | Val Acc: 0.8906 | Val Macro F1: 0.6920
Epoch 04/08 | Loss: 0.1562 | Val Acc: 0.8914 | Val Macro F1: 0.6657
Epoch 05/08 | Loss: 0.1417 | Val Acc: 0.7885 | Val Macro F1: 0.5091
Epoch 06/08 | Loss: 0.1233 | Val Acc: 0.9590 | Val Macro F1: 0.8471
Epoch 07/08 | Loss: 0.1168 | Val Acc: 0.9657 | Val Macro F1: 0.8810


## 4. Comprehensive Comparison Table & Visualizations (ML vs. DL)

In [ ]:
comparison_df = pd.DataFrame(benchmark_results)
display(comparison_df.style.highlight_max(subset=["Test Accuracy", "Weighted F1", "Macro F1"], color="lightgreen")
                           .highlight_min(subset=["Overfit Gap", "Train Time (s)", "Inference (ms/sample)"], color="lightblue"))

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot 1: Macro F1 vs Weighted F1
melted = comparison_df.melt(id_vars=["Model", "Type"], value_vars=["Weighted F1", "Macro F1"], var_name="Metric", value_name="Score")
sns.barplot(data=melted, x="Model", y="Score", hue="Metric", ax=axes[0], palette="Set2")
axes[0].set_title("Macro F1 vs. Weighted F1 (ML vs. DL)", fontsize=13)
axes[0].set_ylim(0.5, 1.05)
axes[0].tick_params(axis="x", rotation=20)
axes[0].legend(loc="lower right")

# Plot 2: Overfitting Diagnosis (Train vs Test Accuracy)
x = np.arange(len(comparison_df))
w = 0.35
axes[1].bar(x - w/2, comparison_df["Train Accuracy"], w, label="Train Accuracy", color="royalblue")
axes[1].bar(x + w/2, comparison_df["Test Accuracy"], w, label="Test Accuracy", color="darkorange")
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison_df["Model"], rotation=20, ha="right")
axes[1].set_ylim(0.7, 1.02)
axes[1].set_title("Train vs. Test Accuracy (Overfitting Gap)", fontsize=13)
axes[1].legend()

# Plot 3: Inference Latency (ms per sample)
sns.barplot(data=comparison_df, x="Model", y="Inference (ms/sample)", hue="Type", dodge=False, ax=axes[2], palette="coolwarm")
axes[2].set_title("Inference Latency (Lower = Faster Deployment)", fontsize=13)
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 5. Detailed Classification Report & Confusion Matrix (Best Model)

In [ ]:
best_idx = comparison_df["Macro F1"].idxmax()
best_name = comparison_df.loc[best_idx, "Model"]
print(f"=== Best Model Overall (by Macro F1): {best_name} ===")

# Determine predictions for best model
if best_name == "Random Forest (Balanced)":
    final_preds = ml_models["Random Forest (Balanced)"].predict(X_test_scaled)
elif best_name == "LightGBM (Balanced)":
    final_preds = ml_models["LightGBM (Balanced)"].predict(X_test_scaled)
elif best_name == "Deep Residual MLP":
    final_preds = mlp_test_preds
else:
    final_preds = cnn_test_preds

print("\nDetailed Per-Class Performance Report:")
print(classification_report(y_test, final_preds, target_names=classes, zero_division=0))

# Normalized Confusion Matrix
cm = confusion_matrix(y_test, final_preds, normalize="true")
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2%", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title(f"Normalized Confusion Matrix — {best_name}", fontsize=13)
plt.ylabel("True Attack Category")
plt.xlabel("Predicted Category")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Save Artifacts to Models Directory

In [ ]:
import joblib

os.makedirs("../Models", exist_ok=True)
joblib.dump(scaler, "../Models/subset_scaler.pkl")
joblib.dump(label_encoder, "../Models/subset_label_encoder.pkl")
comparison_df.to_csv("../Models/subset_comparison_benchmark.csv", index=False)
torch.save(mlp_model.state_dict(), "../Models/deep_residual_mlp.pth")
torch.save(cnn_model.state_dict(), "../Models/conv1d_net.pth")
joblib.dump(ml_models["LightGBM (Balanced)"], "../Models/lightgbm_balanced.pkl")
joblib.dump(ml_models["Random Forest (Balanced)"], "../Models/random_forest_balanced.pkl")

print("Successfully saved all preprocessors, comparison results, and model weights to ../Models/!")